# Stepwise Regression with Joint VIF and P-Value Stopping Criterion

**Author:** Abdulrahman Hussein  
**Supervisor:** Dr. Joseph D. Ortiz  
**Institution:** Kent State University, Department of Earth Sciences

---

## Overview

A focused, advanced framework for **bidirectional stepwise regression** with joint
VIF and p-value stopping criteria, designed for VPCA loadings vs. spectral library analysis.

The algorithm at each iteration:
1. **Forward step**: add the most significant candidate (p < p_enter) that doesn't exceed VIF.
2. **Backward step (p-value)**: remove any variable with p > p_remove.
3. **Backward step (VIF)**: remove any variable with VIF > vif_threshold.
4. Stop when no change occurs in an iteration (converged) or `max_iter` reached.

### Outputs (Excel)
- **Summary**  -  one row per VPC (R², Adj R², materials, max VIF)
- **VIF_Details**  -  per-material rows with VIF, R², Adj R², `Sig_p` (entry p-value), `Std_Beta`
- **IterationHistory**  -  every step taken across all components
- **Descriptive_Stats**  -  z-score sanity check
- **Model_Summary**, **ANOVA**, **Coefficients**  -  SPSS-style incremental tables per VPC
- **Settings**


# Cell 1: 1. Import Libraries

In [ ]:
# Cell 2: 1. Import Libraries  -  Code
import os
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import f as f_dist
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass

warnings.filterwarnings("ignore")

# Try seaborn style; fall back gracefully if seaborn isn't installed
for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot", "default"):
    try:
        plt.style.use(style)
        break
    except Exception:
        continue

print("Libraries imported successfully!")


# Cell 3: 2. Core Helper Functions

VIF, OLS fitting, and metric utilities used by the main class.

In [ ]:
# Cell 4: 2. Core Helper Functions  -  Code
def calculate_vif(X: pd.DataFrame) -> pd.DataFrame:
    """
    Variance Inflation Factor for each column in X (uses statsmodels).
    Returns a DataFrame with columns ['Variable', 'VIF'].
    """
    if X.shape[1] < 2:
        return pd.DataFrame({"Variable": list(X.columns), "VIF": [1.0] * X.shape[1]})
    Xc = sm.add_constant(X)
    rows = []
    for i, col in enumerate(Xc.columns):
        if col == "const":
            continue
        try:
            vif = variance_inflation_factor(Xc.values, i)
        except Exception:
            vif = np.inf
        rows.append({"Variable": col, "VIF": round(float(vif), 3)})
    return pd.DataFrame(rows)


def fit_ols(X: pd.DataFrame, y: pd.Series):
    """Fit an OLS model with an intercept added."""
    return sm.OLS(y, sm.add_constant(X, has_constant="add")).fit()


def get_model_metrics(model) -> Dict[str, float]:
    """Extract a small set of standard metrics from a fitted OLS result."""
    return {
        "r2": float(model.rsquared),
        "adj_r2": float(model.rsquared_adj),
        "aic": float(model.aic),
        "bic": float(model.bic),
        "f_statistic": float(model.fvalue) if model.fvalue is not None else np.nan,
        "f_pvalue": float(model.f_pvalue) if model.f_pvalue is not None else np.nan,
        "n_obs": int(model.nobs),
    }


print("Helper functions defined: calculate_vif, fit_ols, get_model_metrics")


# Cell 5: 3. Iteration Log Data Class

In [ ]:
# Cell 6: 3. Iteration Log Data Class  -  Code
@dataclass
class IterationLog:
    """One row of the stepwise iteration history."""
    step: int
    action: str            # "ADD", "REMOVE_P", "REMOVE_VIF"
    variable: str
    p_value: Optional[float]
    vif: Optional[float]
    r2: float
    n_variables: int
    variables: str         # comma-joined list

    def to_dict(self):
        return {
            "step": self.step,
            "action": self.action,
            "variable": self.variable,
            "p_value": self.p_value,
            "vif": self.vif,
            "r2": self.r2,
            "n_variables": self.n_variables,
            "variables": self.variables,
        }


print("IterationLog dataclass defined.")


# Cell 7: 4. StepwiseVIFRegression Class

A single, focused class implementing the same proven workflow as the v3 learning notebook,
plus extras: entry p-value tracking, standardized coefficient storage, iteration history,
sklearn-style API, and built-in plots.

**Why bidirectional only?** The v3 notebook validated that bidirectional (add + p-remove + VIF-remove)
produces the most reliable results for spectral mixing problems. Forward-only tends to lock in
poor early choices; backward-only is impractical when n_features ≫ n_samples (typical for VPCA).


In [ ]:
# Cell 8: 4. StepwiseVIFRegression Class  -  Code
class StepwiseVIFRegression:
    """
    Bidirectional stepwise regression with joint p-value and VIF stopping criteria.

    Parameters
    ----------
    p_enter : float, default 0.05
        Maximum p-value for ADDING a variable.
    p_remove : float, default 0.10
        Minimum p-value to REMOVE an already-selected variable.
    vif_threshold : float, default 2.0
        Maximum acceptable VIF for any selected variable.
    max_iter : int, default 100
        Hard cap on number of bidirectional iterations.
    verbose : int, default 1
        0 = silent, 1 = step-by-step, 2 = detailed.

    Attributes (after .fit())
    -------------------------
    selected_features_ : list[str]
    model_             : statsmodels OLS results (final fitted model, with const)
    iteration_history_ : list[IterationLog]
    vif_final_         : pd.DataFrame  (Variable, VIF)
    entry_pvalues_     : dict[str, float]   p-value at the moment a variable was ADDed
    standardized_coefs_ : dict[str, float]  final OLS coefficients (excluding const)
    """

    def __init__(self, p_enter: float = 0.05, p_remove: float = 0.10,
                 vif_threshold: float = 2.0, max_iter: int = 100, verbose: int = 1):
        if not (0 < p_enter < 1):
            raise ValueError("p_enter must be in (0, 1)")
        if not (0 < p_remove < 1):
            raise ValueError("p_remove must be in (0, 1)")
        if p_remove < p_enter:
            warnings.warn("p_remove < p_enter is unusual: variables may be removed immediately after entering.")
        if vif_threshold <= 1:
            raise ValueError("vif_threshold must be > 1")

        self.p_enter = p_enter
        self.p_remove = p_remove
        self.vif_threshold = vif_threshold
        self.max_iter = max_iter
        self.verbose = verbose

        self.selected_features_: List[str] = []
        self.model_ = None
        self.iteration_history_: List[IterationLog] = []
        self.vif_final_: Optional[pd.DataFrame] = None
        self.feature_names_: List[str] = []
        self.entry_pvalues_: Dict[str, float] = {}
        self.standardized_coefs_: Dict[str, float] = {}
        self._is_fitted = False

    # --- internal helpers ---
    def _print(self, msg, level=1):
        if self.verbose >= level:
            print(msg)

    def _validate(self, X: pd.DataFrame, y: pd.Series):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame")
        if len(X) != len(y):
            raise ValueError("X and y must have the same number of rows")
        if X.isnull().any().any():
            raise ValueError("X contains NaN values")
        if pd.Series(y).isnull().any():
            raise ValueError("y contains NaN values")
        if X.shape[1] > X.shape[0]:
            warnings.warn(
                f"n_features ({X.shape[1]}) > n_samples ({X.shape[0]}). "
                "Stepwise regression with high-dim data is well-defined but be cautious about overfitting."
            )

    def _log(self, step, action, var, p, vif, current):
        if current:
            r2 = float(fit_ols(self.X_[current], self.y_).rsquared)
        else:
            r2 = 0.0
        self.iteration_history_.append(IterationLog(
            step=step, action=action, variable=var,
            p_value=(round(float(p), 4) if p is not None else None),
            vif=(round(float(vif), 3) if vif is not None else None),
            r2=round(r2, 4),
            n_variables=len(current),
            variables=", ".join(current),
        ))

    # --- public API ---
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "StepwiseVIFRegression":
        self._validate(X, y)
        self.X_ = X
        self.y_ = y
        self.feature_names_ = list(X.columns)

        selected: List[str] = []
        remaining: List[str] = list(X.columns)
        self.iteration_history_ = []
        self.entry_pvalues_ = {}
        self.standardized_coefs_ = {}
        step = 0

        self._print(f"Bidirectional Stepwise | p_enter<{self.p_enter} | p_remove>{self.p_remove} | VIF<{self.vif_threshold}")

        for it in range(self.max_iter):
            changed = False

            # ---- FORWARD: best p-value among remaining ----
            best_p, best_var, best_r2 = 1.0, None, 0.0
            for var in remaining:
                try:
                    m = fit_ols(X[selected + [var]], y)
                    p = float(m.pvalues[var])
                    if p < best_p:
                        best_p, best_var, best_r2 = p, var, float(m.rsquared)
                except Exception:
                    continue

            if best_var is not None and best_p < self.p_enter:
                test = selected + [best_var]
                vif_ok = True
                max_vif = 1.0
                if len(test) > 1:
                    vifs = calculate_vif(X[test])["VIF"]
                    max_vif = float(vifs.max())
                    vif_ok = max_vif <= self.vif_threshold

                if vif_ok:
                    step += 1
                    selected.append(best_var)
                    remaining.remove(best_var)
                    self.entry_pvalues_[best_var] = best_p
                    changed = True
                    self._log(step, "ADD", best_var, best_p, max_vif, selected)
                    self._print(f"Step {step}: + ADD    {best_var:<35} | p={best_p:.4f} | R2={best_r2:.4f} | VIF={max_vif:.2f}")
                else:
                    # Permanently drop this candidate (matches v3 behaviour)
                    remaining.remove(best_var)
                    self._print(f"  SKIP   {best_var:<35} | p={best_p:.4f} but VIF={max_vif:.2f} > {self.vif_threshold}", level=2)

            # ---- BACKWARD by p-value ----
            if selected:
                m = fit_ols(X[selected], y)
                pvals = m.pvalues.drop("const", errors="ignore")
                if len(pvals) > 0:
                    worst_var = pvals.idxmax()
                    worst_p = float(pvals.max())
                    if worst_p > self.p_remove:
                        step += 1
                        selected.remove(worst_var)
                        remaining.append(worst_var)
                        self.entry_pvalues_.pop(worst_var, None)
                        changed = True
                        self._log(step, "REMOVE_P", worst_var, worst_p, None, selected)
                        self._print(f"Step {step}: - REMOVE {worst_var:<35} | p={worst_p:.4f} > {self.p_remove}")

            # ---- BACKWARD by VIF ----
            if len(selected) > 1:
                vif_df = calculate_vif(X[selected])
                worst = vif_df.loc[vif_df["VIF"].idxmax()]
                if float(worst["VIF"]) > self.vif_threshold:
                    step += 1
                    var = worst["Variable"]
                    selected.remove(var)
                    remaining.append(var)
                    self.entry_pvalues_.pop(var, None)
                    changed = True
                    self._log(step, "REMOVE_VIF", var, None, float(worst["VIF"]), selected)
                    self._print(f"Step {step}: - REMOVE {var:<35} | VIF={float(worst['VIF']):.2f} > {self.vif_threshold}")

            if not changed:
                self._print("Converged.")
                break

        self.selected_features_ = selected
        if selected:
            self.model_ = fit_ols(X[selected], y)
            self.vif_final_ = calculate_vif(X[selected]) if len(selected) > 1 \
                              else pd.DataFrame([{"Variable": selected[0], "VIF": 1.0}])
            self.standardized_coefs_ = {
                v: float(self.model_.params[v]) for v in selected if v in self.model_.params.index
            }
        else:
            self.model_ = None
            self.vif_final_ = pd.DataFrame()
        self._is_fitted = True

        self._print(f"\nFinal: {len(selected)} variable(s) selected.")
        if selected:
            self._print(f"  R2={self.model_.rsquared:.4f} | Adj R2={self.model_.rsquared_adj:.4f}")
        return self

    # --- prediction & scoring ---
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        self._check_fitted()
        if self.model_ is None:
            raise ValueError("No features were selected; cannot predict.")
        Xs = sm.add_constant(X[self.selected_features_], has_constant="add")
        return self.model_.predict(Xs)

    def score(self, X: pd.DataFrame, y: pd.Series) -> float:
        """R² on (X, y)."""
        self._check_fitted()
        if self.model_ is None:
            return 0.0
        y_pred = self.predict(X)
        ss_res = float(np.sum((y - y_pred) ** 2))
        ss_tot = float(np.sum((y - np.mean(y)) ** 2))
        return 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

    # --- accessors ---
    def get_iteration_history(self) -> pd.DataFrame:
        return pd.DataFrame([log.to_dict() for log in self.iteration_history_])

    def get_vif(self) -> pd.DataFrame:
        self._check_fitted()
        return self.vif_final_.copy() if self.vif_final_ is not None else pd.DataFrame()

    def get_entry_pvalues(self) -> pd.Series:
        self._check_fitted()
        return pd.Series(self.entry_pvalues_)

    def get_standardized_coefs(self) -> pd.Series:
        self._check_fitted()
        return pd.Series(self.standardized_coefs_)

    def _check_fitted(self):
        if not self._is_fitted:
            raise ValueError("Model has not been fitted. Call .fit() first.")

    # --- text summary ---
    def summary(self) -> str:
        self._check_fitted()
        if self.model_ is None:
            return "No features were selected."
        m = self.model_
        out = []
        out.append("=" * 78)
        out.append("STEPWISE REGRESSION (BIDIRECTIONAL)  -  MODEL SUMMARY")
        out.append("=" * 78)
        out.append(f"p_enter={self.p_enter}  p_remove={self.p_remove}  vif_threshold={self.vif_threshold}")
        out.append("")
        out.append(f"Selected features ({len(self.selected_features_)}):")
        for f in self.selected_features_:
            out.append(f"  - {f}")
        out.append("")
        out.append("-" * 78)
        out.append(f"R-squared        : {m.rsquared:.6f}")
        out.append(f"Adj R-squared    : {m.rsquared_adj:.6f}")
        out.append(f"AIC              : {m.aic:.2f}")
        out.append(f"BIC              : {m.bic:.2f}")
        out.append(f"F-statistic      : {m.fvalue:.4f}")
        out.append(f"F p-value        : {m.f_pvalue:.4e}")
        out.append(f"Observations     : {int(m.nobs)}")
        out.append("-" * 78)
        out.append(f"{'Variable':<28} {'Coef':>10} {'StdErr':>10} {'t':>8} {'P>|t|':>10} {'VIF':>8} {'EntryP':>10}")
        out.append("-" * 78)
        for v in ["const"] + self.selected_features_:
            if v not in m.params.index:
                continue
            coef = float(m.params[v])
            se = float(m.bse[v])
            t = float(m.tvalues[v])
            p = float(m.pvalues[v])
            if v == "const":
                vif_str, ep_str = "N/A", "N/A"
            else:
                vif_row = self.vif_final_[self.vif_final_["Variable"] == v]
                vif_str = f"{float(vif_row['VIF'].values[0]):.2f}" if not vif_row.empty else "N/A"
                ep_str = f"{self.entry_pvalues_.get(v, float('nan')):.4f}" if v in self.entry_pvalues_ else "N/A"
            out.append(f"{v:<28} {coef:>10.4f} {se:>10.4f} {t:>8.3f} {p:>10.4f} {vif_str:>8} {ep_str:>10}")
        out.append("=" * 78)
        return "\n".join(out)

    # --- visualization ---
    def plot_iteration_history(self, figsize=(12, 5)):
        self._check_fitted()
        h = self.get_iteration_history()
        if h.empty:
            print("No iteration history to plot.")
            return
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        axes[0].plot(h["step"], h["r2"], "b-o", linewidth=2, markersize=6)
        axes[0].set_xlabel("Step"); axes[0].set_ylabel("R²")
        axes[0].set_title("R² across selection steps"); axes[0].grid(alpha=0.3)

        colors = {"ADD": "green", "REMOVE_P": "red", "REMOVE_VIF": "orange"}
        bar_colors = [colors.get(a, "gray") for a in h["action"]]
        axes[1].bar(h["step"], h["n_variables"], color=bar_colors, alpha=0.75)
        axes[1].set_xlabel("Step"); axes[1].set_ylabel("# variables in model")
        axes[1].set_title("Variables in model (green=ADD, red=REMOVE_P, orange=REMOVE_VIF)")
        axes[1].grid(alpha=0.3, axis="y")
        plt.tight_layout(); plt.show()

    def plot_vif(self, figsize=(10, 6)):
        self._check_fitted()
        if self.vif_final_ is None or self.vif_final_.empty:
            print("No VIF to plot.")
            return
        df = self.vif_final_.sort_values("VIF")
        fig, ax = plt.subplots(figsize=figsize)
        cols = ["green" if v < self.vif_threshold else "red" for v in df["VIF"]]
        ax.barh(df["Variable"], df["VIF"], color=cols, alpha=0.75)
        ax.axvline(self.vif_threshold, color="red", linestyle="--",
                   label=f"VIF threshold = {self.vif_threshold}")
        ax.axvline(1, color="gray", linestyle=":", label="VIF = 1")
        ax.set_xlabel("VIF"); ax.set_title("VIF for selected features")
        ax.legend(); ax.grid(alpha=0.3, axis="x")
        plt.tight_layout(); plt.show()

    def plot_coefficients(self, figsize=(10, 6)):
        self._check_fitted()
        if self.model_ is None:
            print("No model fitted.")
            return
        coefs = self.model_.params.drop("const", errors="ignore")
        ci = self.model_.conf_int().drop("const", errors="ignore")
        fig, ax = plt.subplots(figsize=figsize)
        errors = [(coefs.values - ci[0].values), (ci[1].values - coefs.values)]
        cols = ["blue" if c > 0 else "red" for c in coefs.values]
        ax.barh(range(len(coefs)), coefs.values, xerr=errors,
                color=cols, alpha=0.75, capsize=4)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_yticks(range(len(coefs))); ax.set_yticklabels(coefs.index)
        ax.set_xlabel("Coefficient"); ax.set_title("Coefficients with 95% CI")
        ax.grid(alpha=0.3, axis="x")
        plt.tight_layout(); plt.show()


print("StepwiseVIFRegression class defined (bidirectional only).")


# Cell 9: 5. SPSS-Style Regression Tables

Generates **Model Summary**, **ANOVA**, and **Coefficients** tables for the incremental
sequence of models built as variables enter the equation. This mirrors the output style
of statistical software like SPSS and is useful for journal-quality reports.


In [ ]:
# Cell 10: 5. SPSS-Style Regression Tables  -  Code
def build_regression_tables(vpc_name: str, y: pd.Series, X: pd.DataFrame,
                            selected: List[str], history: List[IterationLog]
                            ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build SPSS-style Model Summary, ANOVA, and Coefficients tables.
    Each model adds one more variable, in the order the variables entered.
    """
    n = len(y)
    zero_order = {col: round(float(y.corr(X[col])), 4) for col in selected}

    # Order = order in which variables were ADDed
    add_order = [h.variable for h in history if h.action == "ADD"]
    seen = set()
    sel_ordered = []
    for v in add_order:
        if v in selected and v not in seen:
            sel_ordered.append(v)
            seen.add(v)
    # Append any selected vars that didn't show up in history (defensive)
    for v in selected:
        if v not in seen:
            sel_ordered.append(v)

    summary_rows, anova_rows, coef_rows = [], [], []
    prev_r2 = 0.0

    for i, _ in enumerate(sel_ordered):
        vars_in = sel_ordered[: i + 1]
        k = len(vars_in)
        m = fit_ols(X[vars_in], y)
        r2 = float(m.rsquared)
        r2chg = r2 - prev_r2
        df1, df2 = 1, n - k - 1
        if df2 > 0 and (1 - r2) > 0:
            f_chg = (r2chg / df1) / ((1 - r2) / df2)
            sig_f = round(float(f_dist.sf(f_chg, df1, df2)), 4)
        else:
            f_chg, sig_f = None, None

        summary_rows.append({
            "Dependent": vpc_name, "Model": i + 1,
            "R": round(float(np.sqrt(r2)), 4),
            "R Square": round(r2, 4),
            "Adj R Square": round(float(m.rsquared_adj), 4),
            "Std Error": round(float(np.sqrt(m.mse_resid)), 4),
            "R Sq Change": round(r2chg, 4),
            "F Change": round(f_chg, 3) if f_chg is not None else None,
            "df1": df1, "df2": df2,
            "Sig F Change": sig_f,
        })

        ms_reg = float(m.ess) / k
        ms_res = float(m.ssr) / (n - k - 1) if (n - k - 1) > 0 else np.nan
        anova_rows += [
            {"Model": i + 1, "Dependent": vpc_name, "Source": "Regression",
             "Sum of Squares": round(float(m.ess), 4), "df": k,
             "Mean Square": round(ms_reg, 4),
             "F": round(float(m.fvalue), 3), "Sig": round(float(m.f_pvalue), 4)},
            {"Model": i + 1, "Dependent": vpc_name, "Source": "Residual",
             "Sum of Squares": round(float(m.ssr), 4), "df": n - k - 1,
             "Mean Square": round(ms_res, 4), "F": None, "Sig": None},
            {"Model": i + 1, "Dependent": vpc_name, "Source": "Total",
             "Sum of Squares": round(float(m.centered_tss), 4), "df": n - 1,
             "Mean Square": None, "F": None, "Sig": None},
        ]

        vif_map = {r["Variable"]: r["VIF"] for _, r in
                   (calculate_vif(X[vars_in]) if k > 1
                    else pd.DataFrame([{"Variable": vars_in[0], "VIF": 1.0}])).iterrows()}

        for var in vars_in:
            coef_rows.append({
                "Dependent": vpc_name, "Model": i + 1, "Variable": var,
                "Std Beta": round(float(m.params[var]), 4),
                "Std Error": round(float(m.bse[var]), 4),
                "t": round(float(m.tvalues[var]), 3),
                "Sig": round(float(m.pvalues[var]), 4),
                "Zero-order r": zero_order.get(var),
                "VIF": round(float(vif_map.get(var, 1.0)), 3),
            })

        prev_r2 = r2

    return pd.DataFrame(summary_rows), pd.DataFrame(anova_rows), pd.DataFrame(coef_rows)


print("build_regression_tables() defined.")


# Cell 11: 6. VPCAStepwiseAnalysis Class

A high-level wrapper that:
1. Loads VPCA loadings + spectral library, auto-detects wavelength columns, aligns wavelengths.
2. Treats data as **already z-scored** (matches the v3 working flow).
3. Runs `StepwiseVIFRegression` for every VPC component.
4. Produces a comprehensive multi-sheet Excel report.


In [ ]:
# Cell 12: 6. VPCAStepwiseAnalysis Class  -  Code
class VPCAStepwiseAnalysis:
    """End-to-end VPCA + spectral-library stepwise analysis."""

    def __init__(self, p_enter: float = 0.05, p_remove: float = 0.10,
                 vif_threshold: float = 2.0, max_iter: int = 100, verbose: int = 1):
        self.p_enter = p_enter
        self.p_remove = p_remove
        self.vif_threshold = vif_threshold
        self.max_iter = max_iter
        self.verbose = verbose

        self.z_loadings_: Optional[pd.DataFrame] = None
        self.z_library_: Optional[pd.DataFrame] = None
        self.summary_results_: Optional[pd.DataFrame] = None
        self.detailed_results_: Optional[pd.DataFrame] = None
        self.descriptive_stats_: Optional[pd.DataFrame] = None
        self.models_: Dict[str, StepwiseVIFRegression] = {}
        self.regression_tables_: Dict[str, Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]] = {}

    # --- data loading ---
    def load_data(self, loadings_file: str, library_file: str
                  ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Load and align VPCA loadings + spectral library (assumed pre-z-scored)."""

        # Loadings
        loadings = pd.read_csv(loadings_file)
        wav_col = next((c for c in loadings.columns
                        if "centre_nm" in c or "wavelength" in c.lower()),
                       loadings.columns[0])
        loadings = loadings.set_index(wav_col)
        loadings = loadings.drop(["Eigenvalues", "Frac Var", "Extracted Frac Var"],
                                 errors="ignore")
        loadings = loadings[[c for c in loadings.columns if c.startswith("VPC")]]
        loadings = loadings.apply(pd.to_numeric, errors="coerce")
        loadings.index = pd.to_numeric(loadings.index, errors="coerce")
        loadings = loadings[loadings.index.notna()]

        # Library
        library = pd.read_csv(library_file)
        wav_col = next((c for c in library.columns
                        if "centre_nm" in c or "wavelength" in c.lower()),
                       library.columns[0])
        library = library.set_index(wav_col)
        library.index = pd.to_numeric(library.index, errors="coerce")
        library = library[library.index.notna()]
        library = library.apply(pd.to_numeric, errors="coerce")
        library = library.dropna(axis=1, how="all")
        library.columns = [c.replace("_Zdrdl", "").replace(" Zdrdl", "").strip()
                           for c in library.columns]
        drop_cols = [c for c in library.columns
                     if "VPC" in c or c.startswith("IRL") or c == "Band" or "centre_nm" in c]
        library = library.drop(columns=drop_cols, errors="ignore")

        # Align wavelengths
        loadings.index = pd.Index(np.round(loadings.index.values, 1))
        library.index = pd.Index(np.round(library.index.values, 1))
        common = sorted(set(loadings.index) & set(library.index))
        if not common:
            raise ValueError("No common wavelengths between loadings and library.")
        loadings = loadings.loc[common]
        library = library.loc[common]

        # Rename VPC columns -> VPCA_Z1, VPCA_Z2, ...
        loadings.columns = [f"VPCA_Z{i+1}" for i in range(loadings.shape[1])]

        self.z_loadings_ = loadings
        self.z_library_ = library

        # Descriptive stats (sanity check that data is actually z-scored)
        all_z = pd.concat([loadings, library], axis=1)
        rows = []
        for col in all_z.columns:
            mean = float(all_z[col].mean())
            std = float(all_z[col].std(ddof=0))
            rows.append({
                "Variable": col,
                "N": len(all_z),
                "Mean": round(mean, 6),
                "Std Dev": round(std, 6),
                "Z-scored?": "yes" if abs(mean) < 1e-6 and abs(std - 1.0) < 1e-6 else "no",
            })
        self.descriptive_stats_ = pd.DataFrame(rows)

        if self.verbose > 0:
            print(f"Loaded {loadings.shape[1]} VPC components, "
                  f"{library.shape[1]} library materials, "
                  f"{len(common)} common wavelengths.")
            n_zscored = (self.descriptive_stats_["Z-scored?"] == "yes").sum()
            n_total = len(self.descriptive_stats_)
            print(f"Descriptive stats: {n_zscored}/{n_total} variables look z-scored "
                  f"(mean~0, std~1).")

        return loadings, library

    # --- analysis ---
    def run_analysis(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Run stepwise selection for every VPC component and build all tables."""
        if self.z_loadings_ is None or self.z_library_ is None:
            raise ValueError("Call load_data(...) first.")

        summary_rows, detailed_rows = [], []
        self.models_ = {}
        self.regression_tables_ = {}

        for vpc in self.z_loadings_.columns:
            if self.verbose > 0:
                print("\n" + "=" * 70)
                print(f"Analyzing {vpc}")
                print("=" * 70)

            sel = StepwiseVIFRegression(
                p_enter=self.p_enter, p_remove=self.p_remove,
                vif_threshold=self.vif_threshold, max_iter=self.max_iter,
                verbose=self.verbose,
            ).fit(self.z_library_, self.z_loadings_[vpc])
            self.models_[vpc] = sel

            selected = sel.selected_features_
            if selected:
                m = sel.model_
                r2 = float(m.rsquared); adj = float(m.rsquared_adj)
                vif_df = sel.get_vif()
                entry_p = sel.get_entry_pvalues()
                std_b = sel.get_standardized_coefs()

                for var in selected:
                    vif_val = float(vif_df.loc[vif_df["Variable"] == var, "VIF"].values[0]) \
                              if not vif_df.empty else np.nan
                    detailed_rows.append({
                        "Component": vpc, "Material": var,
                        "VIF": round(vif_val, 2),
                        "R_squared": round(r2, 4),
                        "Adj_R_squared": round(adj, 4),
                        "Sig_p": round(float(entry_p.get(var, np.nan)), 4)
                                 if var in entry_p.index else np.nan,
                        "Std_Beta": round(float(std_b.get(var, np.nan)), 4)
                                    if var in std_b.index else np.nan,
                    })

                vif_details = ", ".join(
                    f"{r['Variable']}({r['VIF']:.2f})" for _, r in vif_df.iterrows()
                ) if not vif_df.empty else "N/A"
                summary_rows.append({
                    "Component": vpc, "N_Materials": len(selected),
                    "Materials": ", ".join(selected),
                    "R_squared": round(r2, 4),
                    "Adj_R_squared": round(adj, 4),
                    "Max_VIF": round(float(vif_df["VIF"].max()), 2)
                              if not vif_df.empty else np.nan,
                    "VIF_Details": vif_details,
                })

                # SPSS-style tables
                self.regression_tables_[vpc] = build_regression_tables(
                    vpc_name=vpc, y=self.z_loadings_[vpc], X=self.z_library_,
                    selected=selected, history=sel.iteration_history_,
                )
            else:
                summary_rows.append({
                    "Component": vpc, "N_Materials": 0, "Materials": "None",
                    "R_squared": 0.0, "Adj_R_squared": 0.0,
                    "Max_VIF": np.nan, "VIF_Details": "N/A",
                })

        self.summary_results_ = pd.DataFrame(summary_rows)
        self.detailed_results_ = pd.DataFrame(detailed_rows)
        return self.summary_results_, self.detailed_results_

    def get_iteration_history_all(self) -> pd.DataFrame:
        """Combined iteration history across all components."""
        rows = []
        for vpc, m in self.models_.items():
            h = m.get_iteration_history()
            if not h.empty:
                h = h.copy()
                h.insert(0, "Component", vpc)
                rows.append(h)
        return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    # --- export ---
    def export_results(self, output_dir: str,
                       base_filename: str = "stepwise_results") -> Dict[str, str]:
        """
        Write a complete Excel report:
            - Summary, VIF_Details, IterationHistory, Descriptive_Stats, Settings
            - Per-VPC sheet with Model_Summary / ANOVA / Coefficients (SPSS-style)
        """
        os.makedirs(output_dir, exist_ok=True)
        out = {}

        # CSV summary (quick-look)
        csv_path = os.path.join(output_dir, f"{base_filename}_summary.csv")
        self.summary_results_.to_csv(csv_path, index=False)
        out["csv_summary"] = csv_path

        excel_path = os.path.join(output_dir, f"{base_filename}_detailed.xlsx")
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            self.summary_results_.to_excel(writer, sheet_name="Summary", index=False)

            if self.detailed_results_ is not None and not self.detailed_results_.empty:
                cols = ["Component", "Material", "VIF", "R_squared",
                        "Adj_R_squared", "Sig_p", "Std_Beta"]
                self.detailed_results_[cols].to_excel(
                    writer, sheet_name="VIF_Details", index=False
                )

            hist_all = self.get_iteration_history_all()
            if not hist_all.empty:
                hist_all.to_excel(writer, sheet_name="IterationHistory", index=False)

            if self.descriptive_stats_ is not None:
                self.descriptive_stats_.to_excel(
                    writer, sheet_name="Descriptive_Stats", index=False
                )

            # Per-VPC SPSS-style tables on individual sheets
            for vpc, (ms_df, anova_df, coef_df) in self.regression_tables_.items():
                # Excel sheet names <= 31 chars
                sheet = vpc[:31]
                row = 0
                ms_df.to_excel(writer, sheet_name=sheet, startrow=row, index=False)
                row += len(ms_df) + 3
                anova_df.to_excel(writer, sheet_name=sheet, startrow=row, index=False)
                row += len(anova_df) + 3
                coef_df.to_excel(writer, sheet_name=sheet, startrow=row, index=False)

            settings_df = pd.DataFrame({
                "Parameter": ["p_enter", "p_remove", "vif_threshold", "max_iter", "method"],
                "Value": [self.p_enter, self.p_remove, self.vif_threshold,
                          self.max_iter, "bidirectional"],
                "Description": [
                    "p-value threshold to ADD variable",
                    "p-value threshold to REMOVE variable",
                    "Maximum VIF allowed",
                    "Max bidirectional iterations",
                    "Selection method",
                ],
            })
            settings_df.to_excel(writer, sheet_name="Settings", index=False)

        out["excel"] = excel_path
        if self.verbose > 0:
            print(f"\nWrote: {csv_path}")
            print(f"Wrote: {excel_path}")
        return out

    # --- plotting ---
    def get_model(self, component: str) -> StepwiseVIFRegression:
        return self.models_[component]

    def plot_component(self, component: str, figsize=(12, 8)):
        """Std_Beta + Sig_p bars for one component."""
        if self.detailed_results_ is None:
            raise ValueError("Run analysis first.")
        d = self.detailed_results_
        d = d[d["Component"] == component]
        if d.empty:
            print(f"No selected variables for {component}.")
            return
        fig, axes = plt.subplots(2, 1, figsize=figsize)
        cols = ["blue" if v > 0 else "red" for v in d["Std_Beta"]]
        axes[0].barh(d["Material"], d["Std_Beta"], color=cols, alpha=0.75)
        axes[0].axvline(0, color="black", linewidth=0.5)
        axes[0].set_xlabel("Std_Beta")
        axes[0].set_title(f"{component} - Standardized Coefficients")
        axes[0].grid(alpha=0.3, axis="x")

        axes[1].barh(d["Material"], d["Sig_p"], color="green", alpha=0.75)
        axes[1].axvline(self.p_enter, color="red", linestyle="--",
                        label=f"p_enter={self.p_enter}")
        axes[1].set_xlabel("Sig_p (entry p-value)")
        axes[1].set_title(f"{component} - Entry p-values")
        axes[1].legend(); axes[1].grid(alpha=0.3, axis="x")
        plt.tight_layout(); plt.show()

    def plot_all_components(self, figsize=(14, 6)):
        """R² and N_Materials per component side-by-side."""
        if self.summary_results_ is None:
            raise ValueError("Run analysis first.")
        s = self.summary_results_
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        axes[0].bar(s["Component"], s["R_squared"], color="steelblue", alpha=0.85)
        axes[0].set_ylabel("R²"); axes[0].set_title("R² per component")
        axes[0].grid(alpha=0.3, axis="y"); axes[0].tick_params(axis="x", rotation=30)

        axes[1].bar(s["Component"], s["N_Materials"], color="darkorange", alpha=0.85)
        axes[1].set_ylabel("# materials selected"); axes[1].set_title("Selected materials per component")
        axes[1].grid(alpha=0.3, axis="y"); axes[1].tick_params(axis="x", rotation=30)
        plt.tight_layout(); plt.show()


print("VPCAStepwiseAnalysis class defined.")


# Cell 13: 7. Standalone Helper Functions

Functional shortcuts for users who prefer direct usage over the class.

In [ ]:
# Cell 14: 7. Standalone Helper Functions  -  Code
def run_vpca_stepwise(z_loadings: pd.DataFrame, z_library: pd.DataFrame,
                      p_enter: float = 0.05, p_remove: float = 0.10,
                      vif_threshold: float = 2.0, max_iter: int = 100,
                      verbose: int = 1
                      ) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, StepwiseVIFRegression]]:
    """Run bidirectional stepwise per VPC, return (summary, detailed, models)."""
    summary_rows, detailed_rows, models = [], [], {}

    for vpc in z_loadings.columns:
        if verbose > 0:
            print(f"\n=== {vpc} ===")
        sel = StepwiseVIFRegression(
            p_enter=p_enter, p_remove=p_remove,
            vif_threshold=vif_threshold, max_iter=max_iter, verbose=verbose
        ).fit(z_library, z_loadings[vpc])
        models[vpc] = sel
        chosen = sel.selected_features_

        if chosen:
            m = sel.model_; r2 = float(m.rsquared); adj = float(m.rsquared_adj)
            vif_df = sel.get_vif()
            ep = sel.get_entry_pvalues(); std_b = sel.get_standardized_coefs()
            for v in chosen:
                vif_val = float(vif_df.loc[vif_df["Variable"] == v, "VIF"].values[0]) \
                          if not vif_df.empty else np.nan
                detailed_rows.append({
                    "Component": vpc, "Material": v,
                    "VIF": round(vif_val, 2),
                    "R_squared": round(r2, 4), "Adj_R_squared": round(adj, 4),
                    "Sig_p": round(float(ep.get(v, np.nan)), 4) if v in ep.index else np.nan,
                    "Std_Beta": round(float(std_b.get(v, np.nan)), 4) if v in std_b.index else np.nan,
                })
            summary_rows.append({
                "Component": vpc, "N_Materials": len(chosen),
                "Materials": ", ".join(chosen),
                "R_squared": round(r2, 4), "Adj_R_squared": round(adj, 4),
                "Max_VIF": round(float(vif_df["VIF"].max()), 2) if not vif_df.empty else np.nan,
                "VIF_Details": ", ".join(f"{r['Variable']}({r['VIF']:.2f})"
                                         for _, r in vif_df.iterrows())
                               if not vif_df.empty else "N/A",
            })
        else:
            summary_rows.append({
                "Component": vpc, "N_Materials": 0, "Materials": "None",
                "R_squared": 0.0, "Adj_R_squared": 0.0,
                "Max_VIF": np.nan, "VIF_Details": "N/A",
            })

    return pd.DataFrame(summary_rows), pd.DataFrame(detailed_rows), models


print("Standalone helper run_vpca_stepwise() defined.")


# Cell 15: 8. Example Usage  -  VPCAStepwiseAnalysis

### 8.1 Recommended: `VPCAStepwiseAnalysis` class


In [ ]:
# Cell 16: 8. Example Usage  -  VPCAStepwiseAnalysis  -  Code
# Update these paths to point at your data
loadings_file = r"learning/test-data/test_loadings.csv"
library_file  = r"learning/test-data/test_library.csv"
output_dir    = r"output"

analyzer = VPCAStepwiseAnalysis(
    p_enter=0.05, p_remove=0.10, vif_threshold=2.0, verbose=1
)
analyzer.load_data(loadings_file, library_file)
summary_df, detailed_df = analyzer.run_analysis()

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("DETAILED (first 15 rows)")
print("=" * 70)
print(detailed_df.head(15).to_string(index=False))

files = analyzer.export_results(output_dir)


# Cell 17: 8.2 Visualizations

In [ ]:
# Cell 18: 8.2 Visualizations  -  Code
# Per-component plot (Std_Beta + Sig_p bars)
analyzer.plot_component("VPCA_Z1")

# All-components overview
analyzer.plot_all_components()

# Iteration history & VIF for a specific component
m = analyzer.get_model("VPCA_Z1")
m.plot_iteration_history()
m.plot_vif()
m.plot_coefficients()
print(m.summary())


# Cell 19: 8.3 Functional Alternative

In [ ]:
# Cell 20: 8.3 Functional Alternative  -  Code
# z-scored loadings/library are already in analyzer.z_loadings_, analyzer.z_library_
summary2, detailed2, models2 = run_vpca_stepwise(
    analyzer.z_loadings_, analyzer.z_library_,
    p_enter=0.05, p_remove=0.10, vif_threshold=2.0, verbose=0
)
print(summary2.to_string(index=False))


# Cell 21: 9. Notes & Recommended Thresholds

- **p_enter** = 0.05 (standard 5% significance for inclusion)
- **p_remove** = 0.10 (slightly relaxed to avoid removing borderline variables)
- **vif_threshold** = 2.0 for spectral mixing (strict, prevents library redundancy)
  – use 5.0 for general regression problems where mild collinearity is acceptable.

### What this notebook produces

The Excel file written by `export_results()` contains:

| Sheet | Description |
|---|---|
| Summary | One row per VPC: materials selected, R², Adj R², Max VIF |
| VIF_Details | One row per (component, material) with VIF, R², `Sig_p`, `Std_Beta` |
| IterationHistory | Every ADD/REMOVE_P/REMOVE_VIF step across all VPCs |
| Descriptive_Stats | Mean/Std for every variable (z-score sanity check) |
| VPCA_Z1, VPCA_Z2, ... | SPSS-style Model Summary, ANOVA, Coefficients per VPC |
| Settings | Parameters used for the run |
